# Notebook 01 — Data Pipeline
**Ahead of the Curve: EV Smart Charging and GB Evening Demand Shape**

Fetches, validates, and caches all raw data sources to `../data/`.  
Run this notebook once before any other notebook.  
All subsequent notebooks load from cache.

**Sources**
| Dataset | Source | File |
|---|---|---|
| GB day-ahead prices (N2EX/EPEX) | Elexon BMRS | `prices_raw.parquet` |
| GB half-hourly demand | National Grid ESO | `demand_raw.parquet` |
| Wind actuals + DA forecasts | National Grid ESO | `wind_raw.parquet` |
| Net interconnector flows | National Grid ESO | `interconnector_raw.parquet` |
| Battery storage capacity | ESO Generation Mix | `storage_raw.parquet` |
| EV registrations | DVLA | `ev_registrations.csv` |
| Chargepoint installations | OZEV | `chargepoints.csv` |

Sunset times are computed in Notebook 02 via `astral` (no fetch required).

In [12]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

# Full sample: 2019-01-01 to present
START_DATE = '2019-01-01'
END_DATE   = pd.Timestamp.today().strftime('%Y-%m-%d')

print(f'Pipeline: {START_DATE} → {END_DATE}')
print(f'Data directory: {DATA_DIR.resolve()}')

Pipeline: 2019-01-01 → 2026-04-03
Data directory: /home/ndrew/data


## 1. GB Day-Ahead Prices — Elexon BMRS

Using the Market Index Data (MID) endpoint.  
Aggregated to half-hourly from settlement period data.

In [13]:
PRICE_FILE = DATA_DIR / 'prices_raw.parquet'

def fetch_elexon_prices(start: str, end: str, chunk_days: int = 7) -> pd.DataFrame:
    """
    Fetch N2EX/EPEX day-ahead prices from Elexon Insights API.
    Returns DataFrame with UTC datetime index, column 'price_gbp_mwh'.

    Endpoint: GET /datasets/MID
    Params:   from=<ISO8601>, to=<ISO8601>
    The API accepts a date range via from/to but has an undocumented
    window limit. We use 7-day chunks which is safely within the limit.

    Multiple data providers (APXMIDP, N2EXMIDP) are returned per SP.
    We take the mean across providers per settlement period.

    Docs: https://developer.data.elexon.co.uk/
    """
    base    = 'https://data.elexon.co.uk/bmrs/api/v1/datasets/MID'
    frames  = []
    current = pd.Timestamp(start, tz='UTC')
    end_ts  = pd.Timestamp(end, tz='UTC')
    total_chunks = int(np.ceil((end_ts - current).days / chunk_days))
    chunk_num = 0

    while current < end_ts:
        chunk_end = min(current + pd.Timedelta(days=chunk_days), end_ts)
        params = {
            'from': current.strftime('%Y-%m-%dT%H:%M:%SZ'),
            'to':   chunk_end.strftime('%Y-%m-%dT%H:%M:%SZ'),
        }
        try:
            r = requests.get(base, params=params, timeout=30)
            r.raise_for_status()
            data = r.json().get('data', [])
            if data:
                frames.append(pd.DataFrame(data))
        except requests.HTTPError as e:
            print(f'  Warning: {current.date()} chunk returned {e.response.status_code} — skipping')
        current = chunk_end
        chunk_num += 1
        if chunk_num % 10 == 0:
            print(f'  {chunk_num}/{total_chunks} chunks fetched ({current.date()})...', end='\r')
        time.sleep(0.3)

    if not frames:
        raise ValueError('No price data returned from Elexon')

    prices = pd.concat(frames, ignore_index=True)
    prices.columns = [c.lower() for c in prices.columns]

    # Build UTC datetime from settlementDate + settlementPeriod
    # SP 1 = 00:00–00:30 UTC, so SP 1 starts at 00:00 UTC
    prices['datetime'] = (
        pd.to_datetime(prices['settlementdate'])
        + pd.to_timedelta((prices['settlementperiod'].astype(int) - 1) * 30, unit='min')
    )
    prices['datetime'] = prices['datetime'].dt.tz_localize('UTC')

    # Average across data providers (APXMIDP, N2EXMIDP) per settlement period
    prices = (
        prices.groupby('datetime')['price']
        .mean()
        .rename('price_gbp_mwh')
        .sort_index()
        .to_frame()
    )
    prices = prices.resample('30min').mean()  # ensure uniform 30-min grid
    return prices


if PRICE_FILE.exists():
    print(f'Cache hit: {PRICE_FILE}')
    prices = pd.read_parquet(PRICE_FILE)
else:
    print('Fetching prices from Elexon BMRS (7-day chunks — ~5 min for 6 years)...')
    prices = fetch_elexon_prices(START_DATE, END_DATE)
    prices.to_parquet(PRICE_FILE)
    print(f'\nSaved → {PRICE_FILE}')

print(f'Price data: {len(prices):,} half-hours | {prices.index.min()} → {prices.index.max()}')
print(f'Price range: £{prices.price_gbp_mwh.min():.1f} → £{prices.price_gbp_mwh.max():.1f}/MWh')
prices.head()

Cache hit: ../data/prices_raw.parquet
Price data: 127,155 half-hours | 2019-01-01 00:00:00+00:00 → 2026-04-03 01:00:00+00:00
Price range: £-80.6 → £991.8/MWh


,price_gbp_mwh
datetime,
2019-01-01 00:00:00+00:00,24.405
2019-01-01 00:30:00+00:00,25.120
2019-01-01 01:00:00+00:00,20.950
2019-01-01 01:30:00+00:00,19.660
2019-01-01 02:00:00+00:00,17.045


## 2. Demand, Wind Actuals, Wind Forecasts — National Grid ESO

ESO Data Portal API (CKAN-based).  
Half-hourly settlement data.

In [14]:
WIND_FILE   = DATA_DIR / 'wind_raw.parquet'
DEMAND_FILE = DATA_DIR / 'demand_raw.parquet'

# NESO (formerly ESO) Historic Demand Data
# Domain migrated from api.nationalgrideso.com → api.neso.energy
# Data is split into one CSV per year. Resource IDs from:
# https://www.neso.energy/data-portal/historic-demand-data
NESO_DATASET_ID = '8f2fe0af-871c-488d-8bad-960426f24601'
NESO_YEAR_RESOURCES = {
    2019: 'dd9de980-d724-415a-b344-d8ae11321432',
    2020: '33ba6857-2a55-479f-9308-e5c4c53d4381',
    2021: '18c69c42-f20d-46f0-84e9-e279045befc6',
    2022: 'bb44a1b5-75b1-4db2-8491-257f23385006',
    2023: 'bf5ab335-9b40-4ea4-b93a-ab4af7bce003',
    2024: 'f6d02c0f-957b-48cb-82ee-09003f2ba759',
    2025: 'b2bde559-3455-4021-b179-dfe60c0337b0',
    2026: '8a4a771c-3929-4e56-93ad-cdf13219dea5',
}
# Columns we need (present in every year's CSV)
# SETTLEMENT_DATE, SETTLEMENT_PERIOD, ND, TSD,
# WIND_ACTUAL, WIND_FORECAST (= embedded wind outturn and DA forecast)
NESO_BASE = 'https://api.neso.energy'

def fetch_neso_year(year: int, resource_id: str) -> pd.DataFrame:
    """Download one year of historic demand data as CSV from NESO."""
    url = f'{NESO_BASE}/dataset/{NESO_DATASET_ID}/resource/{resource_id}/download/demanddata_{year}.csv'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    from io import StringIO
    df = pd.read_csv(StringIO(r.text))
    df.columns = [c.strip().upper() for c in df.columns]
    return df

def parse_neso_datetime(df: pd.DataFrame) -> pd.DataFrame:
    """Build UTC datetime index from SETTLEMENT_DATE + SETTLEMENT_PERIOD."""
    date_col = next(c for c in df.columns if 'SETTLEMENT_DATE' in c or 'SETTLEMENTDATE' in c)
    sp_col   = next(c for c in df.columns if 'SETTLEMENT_PERIOD' in c or 'SETTLEMENTPERIOD' in c)
    df['datetime'] = (
        pd.to_datetime(df[date_col])
        + pd.to_timedelta((df[sp_col].astype(int) - 1) * 30, unit='min')
    )
    df['datetime'] = df['datetime'].dt.tz_localize('UTC')
    return df.set_index('datetime').sort_index()


if WIND_FILE.exists() and DEMAND_FILE.exists():
    print('Cache hit: wind and demand files')
    wind   = pd.read_parquet(WIND_FILE)
    demand = pd.read_parquet(DEMAND_FILE)
else:
    start_year = int(START_DATE[:4])
    end_year   = int(END_DATE[:4])
    years      = [y for y in NESO_YEAR_RESOURCES if start_year <= y <= end_year]
    frames = []
    for yr in years:
        rid = NESO_YEAR_RESOURCES[yr]
        print(f'  Fetching {yr}...', end=' ')
        try:
            df = fetch_neso_year(yr, rid)
            frames.append(df)
            print(f'{len(df):,} rows')
        except Exception as e:
            print(f'FAILED: {e}')
        time.sleep(0.5)
    
    raw = pd.concat(frames, ignore_index=True)
    raw = parse_neso_datetime(raw)
    raw = raw.loc[START_DATE:END_DATE]
    print(f'\nTotal rows: {len(raw):,}')
    print('Columns:', list(raw.columns))
    
    # Wind: embedded wind outturn only — transmission wind fetched separately in cell 2b
    wind_cols = [c for c in raw.columns if 'WIND' in c]
    print('Wind columns:', wind_cols)
    wind = raw[wind_cols].copy()
    wind.to_parquet(WIND_FILE)
    print(f'Wind saved → {WIND_FILE}')
    
    # Demand: ND = National Demand, TSD = Transmission System Demand
    demand_cols = [c for c in raw.columns if c in ('ND', 'TSD')]
    print('Demand columns:', demand_cols)
    demand = raw[demand_cols].copy()
    demand.to_parquet(DEMAND_FILE)
    print(f'Demand saved → {DEMAND_FILE}')
    
    # Interconnectors: already in this CSV — extract and save now
    # Saves running the separate Elexon INTERFUELHH fetch
    ic_cols = [c for c in raw.columns if c.endswith('_FLOW')]
    print('Interconnector columns:', ic_cols)
    interconnector = raw[ic_cols].copy()
    interconnector['net_interconnector_mw'] = interconnector[ic_cols].sum(axis=1)
    interconnector.to_parquet(DATA_DIR / 'interconnector_raw.parquet')
    print(f'Interconnector saved → {DATA_DIR}/interconnector_raw.parquet')

print(f'\nWind:   {len(wind):,} rows | cols: {list(wind.columns)}')
print(f'Demand: {len(demand):,} rows | cols: {list(demand.columns)}')
demand.describe()

Cache hit: wind and demand files

Wind:   73,536 rows | cols: ['EMBEDDED_WIND_GENERATION', 'EMBEDDED_WIND_CAPACITY']
Demand: 73,536 rows | cols: ['ND', 'TSD']


,ND,TSD
count,73536.000000,73536.00000
mean,27013.310351,29205.43886
std,6229.953094,5935.99832
min,12803.000000,15297.00000
25%,22176.000000,24733.75000
50%,26176.000000,28334.00000
75%,30964.000000,32819.25000
max,47382.000000,50101.00000


In [16]:
# Extract interconnector flows from the already-downloaded raw data
# Run this once if interconnector_raw.parquet is missing

import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')

# Re-parse the raw demand CSVs that are already cached
# We need to reload raw to get the *_FLOW columns
NESO_DATASET_ID = '8f2fe0af-871c-488d-8bad-960426f24601'
NESO_YEAR_RESOURCES = {
    2019: 'dd9de980-d724-415a-b344-d8ae11321432',
    2020: '33ba6857-2a55-479f-9308-e5c4c53d4381',
    2021: '18c69c42-f20d-46f0-84e9-e279045befc6',
    2022: 'bb44a1b5-75b1-4db2-8491-257f23385006',
    2023: 'bf5ab335-9b40-4ea4-b93a-ab4af7bce003',
    2024: 'f6d02c0f-957b-48cb-82ee-09003f2ba759',
    2025: 'b2bde559-3455-4021-b179-dfe60c0337b0',
    2026: '8a4a771c-3929-4e56-93ad-cdf13219dea5',
}

import requests, time
from io import StringIO

frames = []
for yr, rid in sorted(NESO_YEAR_RESOURCES.items()):
    url = f'https://api.neso.energy/dataset/{NESO_DATASET_ID}/resource/{rid}/download/demanddata_{yr}.csv'
    print(f'  {yr}...', end=' ')
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))
        df.columns = [c.strip().upper() for c in df.columns]
        frames.append(df)
        print(f'{len(df):,} rows')
    except Exception as e:
        print(f'FAILED: {e}')
    time.sleep(0.3)

raw = pd.concat(frames, ignore_index=True)

# Parse datetime
date_col = next(c for c in raw.columns if 'SETTLEMENT_DATE' in c or 'SETTLEMENTDATE' in c)
sp_col   = next(c for c in raw.columns if 'SETTLEMENT_PERIOD' in c or 'SETTLEMENTPERIOD' in c)
raw['datetime'] = (
    pd.to_datetime(raw[date_col])
    + pd.to_timedelta((raw[sp_col].astype(int) - 1) * 30, unit='min')
)
raw['datetime'] = raw['datetime'].dt.tz_localize('UTC')
raw = raw.set_index('datetime').sort_index()

# Extract and save interconnector flows
ic_cols = [c for c in raw.columns if c.endswith('_FLOW')]
print('Interconnector columns:', ic_cols)
interconnector = raw[ic_cols].copy()
interconnector['net_interconnector_mw'] = interconnector[ic_cols].sum(axis=1)
interconnector.to_parquet(DATA_DIR / 'interconnector_raw.parquet')
print(f'Saved → {DATA_DIR}/interconnector_raw.parquet')
print(interconnector[['net_interconnector_mw']].describe())

  2019... 17,520 rows
  2020... 17,568 rows
  2021... 17,520 rows
  2022... 17,520 rows
  2023... 17,520 rows
  2024... 17,568 rows
  2025... 17,520 rows
  2026... 3,456 rows
Interconnector columns: ['IFA_FLOW', 'IFA2_FLOW', 'BRITNED_FLOW', 'MOYLE_FLOW', 'EAST_WEST_FLOW', 'NEMO_FLOW', 'NSL_FLOW', 'ELECLINK_FLOW', 'VIKING_FLOW', 'GREENLINK_FLOW']
Saved → ../data/interconnector_raw.parquet
       net_interconnector_mw
count          126192.000000
mean             2386.714007
std              2651.267851
min             -7914.000000
25%               921.000000
50%              2818.000000
75%              4106.000000
max              9229.000000


## 2b. Interconnector flows

Already extracted from the historic demand CSV above (IFA, IFA2, BritNed, Moyle, etc.).
Verify the file exists and inspect.

In [17]:
INTERCON_FILE = DATA_DIR / 'interconnector_raw.parquet'
interconnector = pd.read_parquet(INTERCON_FILE)
print(f'Interconnector: {len(interconnector):,} half-hours')
print(f'Columns: {list(interconnector.columns)}')
interconnector[['net_interconnector_mw']].describe()

Interconnector: 126,192 half-hours
Columns: ['IFA_FLOW', 'IFA2_FLOW', 'BRITNED_FLOW', 'MOYLE_FLOW', 'EAST_WEST_FLOW', 'NEMO_FLOW', 'NSL_FLOW', 'ELECLINK_FLOW', 'VIKING_FLOW', 'GREENLINK_FLOW', 'net_interconnector_mw']


,net_interconnector_mw
count,126192.000000
mean,2386.714007
std,2651.267851
min,-7914.000000
25%,921.000000
50%,2818.000000
75%,4106.000000
max,9229.000000


## 3. Transmission Wind Actuals + Day-Ahead Forecast — NESO

The historic demand CSV contains **embedded** wind only (small distribution-connected turbines).  
The DiD needs **total GB wind** — and crucially the **day-ahead forecast** so we can compute  
wind forecast error (actual − forecast) as the primary confounder control.

Source: NESO 'Day Ahead Wind Forecast' dataset.
Resource IDs auto-discovered from the CKAN API.

In [19]:
import requests, pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
WIND_TOTAL_FILE = DATA_DIR / 'wind_total_raw.parquet'

# Day Ahead Wind Forecast — NESO
# Dataset: fbe3701d-1487-443e-abe9-47a6c01ecce2
# The archive resource contains all historic national DA wind forecast + outturn
ARCHIVE_URL = (
    'https://api.neso.energy/dataset/fbe3701d-1487-443e-abe9-47a6c01ecce2'
    '/resource/7524ec65-f782-4258-aaf8-5b926c17b966/download/archive_1dayaheadwind.csv'
)

print('Fetching DA wind archive (~1 file, may be large)...')
r = requests.get(ARCHIVE_URL, timeout=300)
r.raise_for_status()

from io import StringIO
wind_raw = pd.read_csv(StringIO(r.text))
wind_raw.columns = [c.strip().upper() for c in wind_raw.columns]
print('Columns:', list(wind_raw.columns))
print(f'Rows: {len(wind_raw):,}')
wind_raw.head(3)

Fetching DA wind archive (~1 file, may be large)...
Columns: ['DATETIME_GMT', 'DATE', 'SETTLEMENT_PERIOD', 'CAPACITY', 'INCENTIVE_FORECAST', 'FORECAST_TIMESTAMP']
Rows: 139,482


,DATETIME_GMT,DATE,SETTLEMENT_PERIOD,CAPACITY,INCENTIVE_FORECAST,FORECAST_TIMESTAMP
0,2018-04-18T23:00:00Z,2018-04-19,1,10973,7632,2018-04-19T08:04:59Z
1,2018-04-18T23:30:00Z,2018-04-19,2,10973,7532,2018-04-19T08:04:59Z
2,2018-04-19T00:00:00Z,2018-04-19,3,10973,7375,2018-04-19T08:04:59Z


In [20]:
import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path('../data')

# Load what we have
wind_embedded = pd.read_parquet(DATA_DIR / 'wind_raw.parquet')  # EMBEDDED_WIND_GENERATION
prices = pd.read_parquet(DATA_DIR / 'prices_raw.parquet')

# Parse the DA forecast archive we just fetched
from io import StringIO
import requests
r = requests.get(
    'https://api.neso.energy/dataset/fbe3701d-1487-443e-abe9-47a6c01ecce2'
    '/resource/7524ec65-f782-4258-aaf8-5b926c17b966/download/archive_1dayaheadwind.csv',
    timeout=300
)
wind_fc = pd.read_csv(StringIO(r.text))
wind_fc.columns = [c.strip().upper() for c in wind_fc.columns]
wind_fc['datetime'] = pd.to_datetime(wind_fc['DATETIME_GMT'], utc=True)
wind_fc = wind_fc.set_index('datetime').sort_index()
wind_fc = wind_fc[~wind_fc.index.duplicated(keep='last')]  # keep latest published forecast per SP

# Align to price index
idx = prices.index
wind_fc_aligned    = wind_fc['INCENTIVE_FORECAST'].reindex(idx, method='nearest')
wind_emb_aligned   = wind_embedded['EMBEDDED_WIND_GENERATION'].reindex(idx, method='nearest')

# Correlation between embedded wind and DA forecast (sanity check)
from scipy.stats import pearsonr
mask = wind_fc_aligned.notna() & wind_emb_aligned.notna()
r_val, _ = pearsonr(wind_fc_aligned[mask], wind_emb_aligned[mask])
print(f'Pearson r (embedded actual vs DA forecast): {r_val:.3f}')
print(f'DA forecast rows: {wind_fc_aligned.notna().sum():,}')
print(f'Embedded actual rows: {wind_emb_aligned.notna().sum():,}')
print(f'Overlap: {mask.sum():,}')

ValueError: cannot reindex on an axis with duplicate labels

In [21]:
import pandas as pd, numpy as np
from pathlib import Path
from io import StringIO
import requests
from scipy.stats import pearsonr

DATA_DIR = Path('../data')

wind_embedded = pd.read_parquet(DATA_DIR / 'wind_raw.parquet')
prices = pd.read_parquet(DATA_DIR / 'prices_raw.parquet')

# Deduplicate both series before aligning
wind_embedded = wind_embedded[~wind_embedded.index.duplicated(keep='first')]
prices = prices[~prices.index.duplicated(keep='first')]

# Fetch DA forecast
print('Fetching DA wind archive...')
r = requests.get(
    'https://api.neso.energy/dataset/fbe3701d-1487-443e-abe9-47a6c01ecce2'
    '/resource/7524ec65-f782-4258-aaf8-5b926c17b966/download/archive_1dayaheadwind.csv',
    timeout=300
)
wind_fc = pd.read_csv(StringIO(r.text))
wind_fc.columns = [c.strip().upper() for c in wind_fc.columns]
wind_fc['datetime'] = pd.to_datetime(wind_fc['DATETIME_GMT'], utc=True)
wind_fc = wind_fc.set_index('datetime').sort_index()
wind_fc = wind_fc[~wind_fc.index.duplicated(keep='last')]

# Align to price index
idx = prices.index
wind_fc_aligned  = wind_fc['INCENTIVE_FORECAST'].reindex(idx, method='nearest')
wind_emb_aligned = wind_embedded['EMBEDDED_WIND_GENERATION'].reindex(idx, method='nearest')

# Correlation
mask = wind_fc_aligned.notna() & wind_emb_aligned.notna()
r_val, _ = pearsonr(wind_fc_aligned[mask], wind_emb_aligned[mask])
print(f'Pearson r (embedded actual vs DA forecast): {r_val:.3f}')
print(f'Overlap: {mask.sum():,} half-hours')

# Also show the date range covered by the DA forecast
print(f'\nDA forecast range: {wind_fc.index.min().date()} → {wind_fc.index.max().date()}')
print(f'Embedded wind range: {wind_embedded.index.min().date()} → {wind_embedded.index.max().date()}')

Fetching DA wind archive...
Pearson r (embedded actual vs DA forecast): 0.576
Overlap: 127,155 half-hours

DA forecast range: 2018-04-16 → 2026-04-04
Embedded wind range: 2021-01-01 → 2026-03-13


In [22]:
import requests, pandas as pd
from io import StringIO

# B1630: Actual or Estimated Wind and Solar Power Generation
# One settlement date at a time, returns all fuel types including wind
r = requests.get(
    'https://data.elexon.co.uk/bmrs/api/v1/datasets/B1630',
    params={'from': '2024-01-15T00:00:00Z', 'to': '2024-01-15T23:30:00Z'},
    timeout=30
)
print(r.status_code)
if r.status_code == 200:
    d = r.json()
    print('Keys:', list(d.keys()))
    print('Sample rows:')
    for row in d['data'][:6]:
        print(' ', row)

404


In [23]:
import requests

base = 'https://data.elexon.co.uk/bmrs/api/v1/datasets/'
params = {'from': '2024-01-15T00:00:00Z', 'to': '2024-01-15T23:30:00Z'}

for endpoint in ['AGWS', 'FUELHH', 'WINDFOR']:
    r = requests.get(base + endpoint, params=params, timeout=30)
    print(f'{r.status_code}  {endpoint}', end='')
    if r.status_code == 200:
        data = r.json().get('data', [])
        print(f'  {len(data)} rows  |  sample: {data[0] if data else "empty"}')
    else:
        print()

404  AGWS
200  FUELHH  820 rows  |  sample: {'dataset': 'FUELHH', 'publishTime': '2026-04-03T19:30:00Z', 'startTime': '2026-04-03T19:00:00Z', 'settlementDate': '2026-04-03', 'settlementPeriod': 41, 'fuelType': 'BIOMASS', 'generation': 1288}
200  WINDFOR  73 rows  |  sample: {'dataset': 'WINDFOR', 'publishTime': '2026-04-03T19:30:00Z', 'startTime': '2026-04-02T20:00:00Z', 'generation': 10554}


In [24]:
import requests, pandas as pd
from io import StringIO

r = requests.get(
    'https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH',
    params={'from': '2024-01-15T00:00:00Z', 'to': '2024-01-21T23:30:00Z'},
    timeout=30
)
print(r.status_code, f'{len(r.json()["data"]):,} rows')

df = pd.DataFrame(r.json()['data'])
print('Fuel types:', sorted(df['fuelType'].unique()))
wind = df[df['fuelType'] == 'WIND']
print(f'\nWIND rows: {len(wind)}')
print(wind[['startTime','settlementPeriod','generation']].head(6).to_string())

200 820 rows
Fuel types: ['BIOMASS', 'CCGT', 'COAL', 'INTELEC', 'INTEW', 'INTFR', 'INTGRNL', 'INTIFA2', 'INTIRL', 'INTNED', 'INTNEM', 'INTNSL', 'INTVKL', 'NPSHYD', 'NUCLEAR', 'OCGT', 'OIL', 'OTHER', 'PS', 'WIND']

WIND rows: 41
                startTime  settlementPeriod  generation
19   2026-04-03T19:00:00Z                41       14814
39   2026-04-03T18:30:00Z                40       15120
59   2026-04-03T18:00:00Z                39       14839
79   2026-04-03T17:30:00Z                38       14050
99   2026-04-03T17:00:00Z                37       14488
119  2026-04-03T16:30:00Z                36       14775


In [25]:
import requests, pandas as pd

r = requests.get(
    'https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH',
    params={'from': '2024-01-15T00:00:00Z', 'to': '2024-01-21T23:30:00Z'},
    timeout=30
)
df = pd.DataFrame(r.json()['data'])
df['startTime'] = pd.to_datetime(df['startTime'])
print('Date range in response:', df['startTime'].min(), '→', df['startTime'].max())
print('Total rows:', len(df))
wind = df[df['fuelType']=='WIND']
print('WIND rows:', len(wind))
print(wind[['startTime','generation']].head(6).to_string())

Date range in response: 2026-04-02 23:00:00+00:00 → 2026-04-03 19:00:00+00:00
Total rows: 820
WIND rows: 41
                    startTime  generation
19  2026-04-03 19:00:00+00:00       14814
39  2026-04-03 18:30:00+00:00       15120
59  2026-04-03 18:00:00+00:00       14839
79  2026-04-03 17:30:00+00:00       14050
99  2026-04-03 17:00:00+00:00       14488
119 2026-04-03 16:30:00+00:00       14775


In [26]:
import requests

r = requests.get(
    'https://api.neso.energy/api/3/action/package_show?id=historic-generation-mix',
    timeout=30
)
resources = r.json()['result']['resources']
for res in sorted(resources, key=lambda x: x.get('name','')):
    print(res['id'], res.get('name',''))

f93d1835-75bc-43e5-84ad-12472b180a98 Historic GB Generation Mix


In [27]:
import requests, pandas as pd
from io import StringIO

# First just get 5 rows to see columns
r = requests.get(
    'https://api.neso.energy/api/3/action/datastore_search',
    params={'resource_id': 'f93d1835-75bc-43e5-84ad-12472b180a98', 'limit': 5},
    timeout=30
)
print(r.status_code)
result = r.json()['result']
print('Total rows:', result['total'])
print('Columns:', [f['id'] for f in result['fields']])
print()
for row in result['records']:
    print(row)

200
Total rows: 302486
Columns: ['_id', 'DATETIME', 'GAS', 'COAL', 'NUCLEAR', 'WIND', 'WIND_EMB', 'HYDRO', 'IMPORTS', 'BIOMASS', 'OTHER', 'SOLAR', 'STORAGE', 'GENERATION', 'CARBON_INTENSITY', 'LOW_CARBON', 'ZERO_CARBON', 'RENEWABLE', 'FOSSIL', 'GAS_perc', 'COAL_perc', 'NUCLEAR_perc', 'WIND_perc', 'WIND_EMB_perc', 'HYDRO_perc', 'IMPORTS_perc', 'BIOMASS_perc', 'OTHER_perc', 'SOLAR_perc', 'STORAGE_perc', 'GENERATION_perc', 'LOW_CARBON_perc', 'ZERO_CARBON_perc', 'RENEWABLE_perc', 'FOSSIL_perc']

{'_id': 7259, 'DATETIME': '2009-06-01T05:00:00', 'GAS': 13916.0, 'COAL': 4431.0, 'NUCLEAR': 7714.0, 'WIND': 10.0, 'WIND_EMB': 2.0, 'HYDRO': 54, 'IMPORTS': 1334.0, 'BIOMASS': 0.0, 'OTHER': 0.0, 'SOLAR': 0.0, 'STORAGE': 0, 'GENERATION': 27462.0, 'CARBON_INTENSITY': 353.0, 'LOW_CARBON': 7780.0, 'ZERO_CARBON': 7778.0, 'RENEWABLE': 66.0, 'FOSSIL': 18347.0, 'GAS_perc': 50.7, 'COAL_perc': 16.1, 'NUCLEAR_perc': 28.1, 'WIND_perc': 0.0, 'WIND_EMB_perc': 0.0, 'HYDRO_perc': 0.2, 'IMPORTS_perc': 4.9, 'BIOMASS_p

In [28]:
import requests, pandas as pd, time
from pathlib import Path

DATA_DIR = Path('../data')
WIND_TOTAL_FILE = DATA_DIR / 'wind_total_raw.parquet'

BASE    = 'https://api.neso.energy/api/3/action/datastore_search'
RES_ID  = 'f93d1835-75bc-43e5-84ad-12472b180a98'
LIMIT   = 10000

frames, offset, total = [], 0, None
while total is None or offset < total:
    r = requests.get(BASE, params={'resource_id': RES_ID, 'limit': LIMIT, 'offset': offset}, timeout=60)
    r.raise_for_status()
    result = r.json()['result']
    if total is None:
        total = result['total']
        print(f'Total rows: {total:,}')
    records = result['records']
    if not records:
        break
    frames.append(pd.DataFrame(records))
    offset += len(records)
    print(f'  {offset:,}/{total:,}', end='\r')
    time.sleep(0.5)

print()
genmix = pd.concat(frames, ignore_index=True)

# Parse datetime — GB local time, convert to UTC
genmix['datetime'] = (
    pd.to_datetime(genmix['DATETIME'])
      .dt.tz_localize('Europe/London', ambiguous='infer', nonexistent='shift_forward')
      .dt.tz_convert('UTC')
)
genmix = genmix.set_index('datetime').sort_index()
genmix = genmix[~genmix.index.duplicated(keep='first')]

# Total wind = transmission + embedded
genmix['WIND_TOTAL'] = genmix['WIND'].astype(float) + genmix['WIND_EMB'].astype(float)

# Also keep STORAGE for NB03 falsification test
wind_total = genmix[['WIND_TOTAL', 'WIND', 'WIND_EMB', 'STORAGE']].copy()

# Filter to our sample period
wind_total = wind_total.loc['2019-01-01':'2026-12-31']
wind_total.to_parquet(WIND_TOTAL_FILE)
print(f'Saved → {WIND_TOTAL_FILE}')
print(f'Rows: {len(wind_total):,}  |  {wind_total.index.min().date()} → {wind_total.index.max().date()}')
print(wind_total.describe().round(1))

Total rows: 302,486
  302,486/302,486


AmbiguousTimeError: 2009-10-25 01:00:00

In [29]:
# Parse datetime — replace the tz_localize line only
genmix['datetime'] = (
    pd.to_datetime(genmix['DATETIME'])
      .dt.tz_localize('Europe/London', ambiguous='NaT', nonexistent='shift_forward')
      .dt.tz_convert('UTC')
)

# Drop the ambiguous clock-change rows (1 hour per year, ~2 rows each)
genmix = genmix.dropna(subset=['datetime'])

genmix = genmix.set_index('datetime').sort_index()
genmix = genmix[~genmix.index.duplicated(keep='first')]

# Total wind = transmission + embedded
genmix['WIND_TOTAL'] = genmix['WIND'].astype(float) + genmix['WIND_EMB'].astype(float)

# Keep STORAGE for NB03 falsification test
wind_total = genmix[['WIND_TOTAL', 'WIND', 'WIND_EMB', 'STORAGE']].copy()
wind_total = wind_total.loc['2019-01-01':'2026-12-31']
wind_total.to_parquet(WIND_TOTAL_FILE)
print(f'Saved → {WIND_TOTAL_FILE}')
print(f'Rows: {len(wind_total):,}  |  {wind_total.index.min().date()} → {wind_total.index.max().date()}')
print(wind_total.describe().round(1))

Saved → ../data/wind_total_raw.parquet
Rows: 127,160  |  2019-01-01 → 2026-04-03
       WIND_TOTAL      WIND  WIND_EMB   STORAGE
count    127160.0  127160.0  127160.0  127160.0
mean       8543.4    6822.3    1721.2     191.2
std        5146.2    4140.1    1085.5     357.8
min         221.0       0.0     125.0       0.0
25%        4269.0    3344.0     877.0       0.0
50%        7727.0    6241.0    1453.0       0.0
75%       12307.0    9963.0    2319.0     220.0
max       23880.0   18439.0    5947.0    2444.0


In [30]:
import requests, pandas as pd
from io import StringIO
from pathlib import Path

DATA_DIR = Path('../data')

# Load wind actuals we just saved
wind_total = pd.read_parquet(DATA_DIR / 'wind_total_raw.parquet')

# Load DA forecast archive
print('Fetching DA wind forecast archive...')
r = requests.get(
    'https://api.neso.energy/dataset/fbe3701d-1487-443e-abe9-47a6c01ecce2'
    '/resource/7524ec65-f782-4258-aaf8-5b926c17b966/download/archive_1dayaheadwind.csv',
    timeout=300
)
r.raise_for_status()
wind_fc = pd.read_csv(StringIO(r.text))
wind_fc.columns = [c.strip().upper() for c in wind_fc.columns]

# Parse datetime — this one is already UTC (ends in Z)
wind_fc['datetime'] = pd.to_datetime(wind_fc['DATETIME_GMT'], utc=True)
wind_fc = wind_fc.set_index('datetime').sort_index()
wind_fc = wind_fc[~wind_fc.index.duplicated(keep='last')]  # keep latest published forecast per SP

print(f'DA forecast: {len(wind_fc):,} rows  |  {wind_fc.index.min().date()} → {wind_fc.index.max().date()}')
print('Columns:', list(wind_fc.columns))

# Align and merge
idx = wind_total.index
fc_aligned = wind_fc['INCENTIVE_FORECAST'].reindex(idx, method='nearest', tolerance=pd.Timedelta('30min'))

wind_total['wind_da_forecast_mw'] = fc_aligned
wind_total['wind_forecast_error']  = wind_total['WIND_TOTAL'] - wind_total['wind_da_forecast_mw']

# Coverage check
covered = wind_total['wind_forecast_error'].notna().sum()
print(f'\nForecast error coverage: {covered:,}/{len(wind_total):,} ({covered/len(wind_total):.1%})')
print(wind_total[['WIND_TOTAL','wind_da_forecast_mw','wind_forecast_error']].dropna().describe().round(1))

# Save updated file
wind_total.to_parquet(DATA_DIR / 'wind_total_raw.parquet')
print(f'\nSaved → {DATA_DIR}/wind_total_raw.parquet')

Fetching DA wind forecast archive...
DA forecast: 139,482 rows  |  2018-04-16 → 2026-04-04
Columns: ['DATETIME_GMT', 'DATE', 'SETTLEMENT_PERIOD', 'CAPACITY', 'INCENTIVE_FORECAST', 'FORECAST_TIMESTAMP']

Forecast error coverage: 126,974/127,160 (99.9%)
       WIND_TOTAL  wind_da_forecast_mw  wind_forecast_error
count    126974.0             126974.0             126974.0
mean       8543.2               6797.2               1746.1
std        5146.6               4422.6               1739.3
min         221.0                233.0              -5635.0
25%        4270.0               3087.0                538.0
50%        7723.0               5989.0               1535.0
75%       12304.0               9875.0               2797.0
max       23880.0              20541.0              11417.0

Saved → ../data/wind_total_raw.parquet


In [18]:
WIND_TOTAL_FILE = DATA_DIR / 'wind_total_raw.parquet'

if WIND_TOTAL_FILE.exists():
    print(f'Cache hit: {WIND_TOTAL_FILE}')
    wind_total = pd.read_parquet(WIND_TOTAL_FILE)
else:
    # Discover resource IDs for the day-ahead wind forecast dataset
    print('Discovering NESO day-ahead wind forecast resources...')
    r = requests.get(
        'https://api.neso.energy/api/3/action/package_show?id=day-ahead-wind-forecast',
        timeout=30
    )
    r.raise_for_status()
    resources = r.json()['result']['resources']
    for res in sorted(resources, key=lambda x: x.get('name', '')):
        print(f"  {res['id']}  {res.get('name', '')}")

    # Filter to annual resources covering our date range
    year_resources = [
        res for res in resources
        if any(str(y) in res.get('name', '') for y in range(2019, 2027))
        and 'FAQ' not in res.get('name', '')
    ]
    print(f'\nFetching {len(year_resources)} year files...')

    frames = []
    for res in sorted(year_resources, key=lambda x: x.get('name', '')):
        url  = res.get('url') or res.get('path', '')
        name = res.get('name', '')
        print(f'  {name}...', end=' ')
        try:
            df = pd.read_csv(url)
            frames.append(df)
            print(f'{len(df):,} rows')
        except Exception as e:
            print(f'FAILED: {e}')
        time.sleep(0.5)

    if not frames:
        raise ValueError('No wind forecast data fetched. Check NESO dataset ID above.')

    wind_raw = pd.concat(frames, ignore_index=True)
    wind_raw.columns = [c.strip().upper() for c in wind_raw.columns]
    print('\nWind columns:', list(wind_raw.columns))

    wind_raw = parse_neso_datetime(wind_raw)
    wind_raw = wind_raw.loc[START_DATE:END_DATE]

    # Identify outturn and forecast columns
    outturn_col  = next((c for c in wind_raw.columns if 'OUTTURN' in c or 'ACTUAL' in c), None)
    forecast_col = next((c for c in wind_raw.columns if 'FORECAST' in c or 'DA' in c), None)
    print(f'Outturn column:  {outturn_col}')
    print(f'Forecast column: {forecast_col}')

    if outturn_col and forecast_col:
        wind_total = wind_raw[[outturn_col, forecast_col]].rename(columns={
            outturn_col:  'wind_actual_mw',
            forecast_col: 'wind_da_forecast_mw',
        })
        wind_total['wind_forecast_error'] = (
            wind_total['wind_actual_mw'] - wind_total['wind_da_forecast_mw']
        )
    else:
        # Fallback: keep all columns, NB02 will compute error
        wind_total = wind_raw.copy()

    wind_total.to_parquet(WIND_TOTAL_FILE)
    print(f'Saved → {WIND_TOTAL_FILE}')

print(f'\nTransmission wind: {len(wind_total):,} rows | cols: {list(wind_total.columns)}')
wind_total.describe()

Discovering NESO day-ahead wind forecast resources...
  90e581e9-2d7f-4eaa-8fe0-6e323ee0f5f7  Day Ahead Wind BMU Forecast
  b2f03146-f05d-4824-a663-3a4f36090c71  Day Ahead Wind Forecast
  7524ec65-f782-4258-aaf8-5b926c17b966  Historic Day Ahead Wind Forecasts

Fetching 0 year files...


ValueError: No wind forecast data fetched. Check NESO dataset ID above.

In [31]:
import requests, pandas as pd
from io import StringIO
from pathlib import Path

DATA_DIR = Path('../data')

# Load wind actuals we just saved
wind_total = pd.read_parquet(DATA_DIR / 'wind_total_raw.parquet')

# Load DA forecast archive
print('Fetching DA wind forecast archive...')
r = requests.get(
    'https://api.neso.energy/dataset/fbe3701d-1487-443e-abe9-47a6c01ecce2'
    '/resource/7524ec65-f782-4258-aaf8-5b926c17b966/download/archive_1dayaheadwind.csv',
    timeout=300
)
r.raise_for_status()
wind_fc = pd.read_csv(StringIO(r.text))
wind_fc.columns = [c.strip().upper() for c in wind_fc.columns]

# Parse datetime — this one is already UTC (ends in Z)
wind_fc['datetime'] = pd.to_datetime(wind_fc['DATETIME_GMT'], utc=True)
wind_fc = wind_fc.set_index('datetime').sort_index()
wind_fc = wind_fc[~wind_fc.index.duplicated(keep='last')]  # keep latest published forecast per SP

print(f'DA forecast: {len(wind_fc):,} rows  |  {wind_fc.index.min().date()} → {wind_fc.index.max().date()}')
print('Columns:', list(wind_fc.columns))

# Align and merge
idx = wind_total.index
fc_aligned = wind_fc['INCENTIVE_FORECAST'].reindex(idx, method='nearest', tolerance=pd.Timedelta('30min'))

wind_total['wind_da_forecast_mw'] = fc_aligned
wind_total['wind_forecast_error']  = wind_total['WIND_TOTAL'] - wind_total['wind_da_forecast_mw']

# Coverage check
covered = wind_total['wind_forecast_error'].notna().sum()
print(f'\nForecast error coverage: {covered:,}/{len(wind_total):,} ({covered/len(wind_total):.1%})')
print(wind_total[['WIND_TOTAL','wind_da_forecast_mw','wind_forecast_error']].dropna().describe().round(1))

# Save updated file
wind_total.to_parquet(DATA_DIR / 'wind_total_raw.parquet')
print(f'\nSaved → {DATA_DIR}/wind_total_raw.parquet')

Fetching DA wind forecast archive...
DA forecast: 139,482 rows  |  2018-04-16 → 2026-04-04
Columns: ['DATETIME_GMT', 'DATE', 'SETTLEMENT_PERIOD', 'CAPACITY', 'INCENTIVE_FORECAST', 'FORECAST_TIMESTAMP']

Forecast error coverage: 126,974/127,160 (99.9%)
       WIND_TOTAL  wind_da_forecast_mw  wind_forecast_error
count    126974.0             126974.0             126974.0
mean       8543.2               6797.2               1746.1
std        5146.6               4422.6               1739.3
min         221.0                233.0              -5635.0
25%        4270.0               3087.0                538.0
50%        7723.0               5989.0               1535.0
75%       12304.0               9875.0               2797.0
max       23880.0              20541.0              11417.0

Saved → ../data/wind_total_raw.parquet


In [32]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../data')
files = {
    'prices_raw.parquet':       'GB day-ahead prices (Elexon MID)',
    'demand_raw.parquet':       'GB half-hourly demand ND/TSD (NESO)',
    'wind_raw.parquet':         'Embedded wind generation (NESO)',
    'wind_total_raw.parquet':   'Total wind actuals + DA forecast error (NESO genmix + DA archive)',
    'interconnector_raw.parquet': 'Net interconnector flows (NESO historic demand)',
    'chargepoints.parquet':     'Compliant chargepoint stock (OZEV)',
    'ev_registrations.parquet': 'BEV registrations (DVLA)',
}

print('=== DATA INVENTORY ===\n')
for fname, desc in files.items():
    fpath = DATA_DIR / fname
    if fpath.exists():
        df = pd.read_parquet(fpath)
        size_mb = fpath.stat().st_size / 1e6
        print(f'  ✓  {fname:<35} {len(df):>8,} rows  {size_mb:.1f} MB')
        print(f'       {desc}')
    else:
        print(f'  ✗  {fname:<35} MISSING')
        print(f'       {desc}')

=== DATA INVENTORY ===

  ✓  prices_raw.parquet                   127,155 rows  1.5 MB
       GB day-ahead prices (Elexon MID)
  ✓  demand_raw.parquet                    73,536 rows  1.1 MB
       GB half-hourly demand ND/TSD (NESO)
  ✓  wind_raw.parquet                      73,536 rows  0.8 MB
       Embedded wind generation (NESO)
  ✓  wind_total_raw.parquet               127,160 rows  2.7 MB
       Total wind actuals + DA forecast error (NESO genmix + DA archive)
  ✓  interconnector_raw.parquet           126,192 rows  2.5 MB
       Net interconnector flows (NESO historic demand)
  ✗  chargepoints.parquet                MISSING
       Compliant chargepoint stock (OZEV)
  ✗  ev_registrations.parquet            MISSING
       BEV registrations (DVLA)


In [35]:
!pip install odfpy -q

In [38]:
import pandas as pd
from pathlib import Path

# Point this at wherever you downloaded the ODS file
ODS_PATH = Path.home() / 'Ahead-of-the-curve' / 'data' / 'veh0141.ods'  # adjust filename if different

# List the sheet names first
xl = pd.ExcelFile(ODS_PATH, engine='odf')
print('Sheets:', xl.sheet_names)

Sheets: ['Cover_sheet', 'Contents', 'Notes', 'VEH0141a_Fuel', 'VEH0141b_GenModels']


In [41]:
ODS_PATH = Path.home() / 'Ahead-of-the-curve' / 'data' / 'veh0141.ods'  # adjust if filename differs

df = pd.read_ods(ODS_PATH, sheet_name='VEH0141a_Fuel', header=None)
print(df.shape)
print(df.iloc[:10, :8].to_string())

AttributeError: module 'pandas' has no attribute 'read_ods'

In [42]:
from pathlib import Path
import pandas as pd

ODS_PATH = Path.home() / 'Ahead-of-the-curve' / 'data' / 'veh0141.ods'

df = pd.read_excel(ODS_PATH, sheet_name='VEH0141a_Fuel', header=None, engine='odf')
print(df.shape)
print(df.iloc[:10, :8].to_string())

(2399, 9)
                                                                                                                                                                    0                        1       2                  3                 4                                 5                                 6                        7
0  Licensed plug-in vehicles (PiVs) [note 1] at the end of the quarter by body type [note 2] and fuel type, Great Britain and United Kingdom from 2010 Q1 (end March)                      NaN     NaN                NaN               NaN                               NaN                               NaN                      NaN
1                                                                                                                                                      Table VEH0141a                      NaN     NaN                NaN               NaN                               NaN                               NaN                      NaN
2  

In [43]:
import numpy as np

# Read with correct header row
df = pd.read_excel(ODS_PATH, sheet_name='VEH0141a_Fuel', header=4, engine='odf')
df.columns = [str(c).strip() for c in df.columns]
print('Columns:', list(df.columns))
print('Geographies:', df['Geography'].dropna().unique())
print('BodyTypes:', df['BodyType'].dropna().unique())

Columns: ['Geography', 'Date', 'Units', 'BodyType', 'Battery electric', 'Plug-in hybrid electric - petrol', 'Plug-in hybrid electric - diesel', 'Range extended electric', 'Total']
Geographies: ['England' 'Scotland' 'Wales' 'Great Britain' 'Northern Ireland'
 'United Kingdom']
BodyTypes: ['Buses and coaches' 'Cars' 'Heavy goods vehicles' 'Light goods vehicles'
 'Motorcycles' 'Other vehicles' 'Total']


In [44]:
# Filter to United Kingdom, Cars, Battery electric only
bev = df[
    (df['Geography'] == 'United Kingdom') &
    (df['BodyType'] == 'Cars')
][['Date', 'Battery electric']].copy()

# Replace '[low]' suppressed values with NaN, drop them
bev['Battery electric'] = pd.to_numeric(bev['Battery electric'], errors='coerce')
bev = bev.dropna()

# Parse quarter strings → period end date
# Format: "2019 Q1 (end March)" → extract "2019 Q1"
bev['quarter'] = bev['Date'].str.extract(r'(\d{4} Q\d)')
bev['date'] = pd.PeriodIndex(bev['quarter'], freq='Q').to_timestamp('Q')
bev = bev.set_index('date').sort_index()
bev = bev.rename(columns={'Battery electric': 'bev_cumulative'})

print(f'BEV series: {len(bev)} quarters')
print(f'Range: {bev.index.min().date()} → {bev.index.max().date()}')
print(bev.tail(8).to_string())

DateParseError: Unknown datetime string format, unable to parse: 2014 Q3

In [45]:
def parse_dvla_quarter(s):
    # "2019 Q1" → year=2019, q=1 → end of quarter date
    year, q = int(s[:4]), int(s[-1])
    month = q * 3  # Q1→3, Q2→6, Q3→9, Q4→12
    return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)

bev['date'] = bev['quarter'].apply(parse_dvla_quarter)
bev = bev.set_index('date').sort_index()
bev = bev.rename(columns={'Battery electric': 'bev_cumulative'})

print(f'BEV series: {len(bev)} quarters')
print(f'Range: {bev.index.min().date()} → {bev.index.max().date()}')
print(bev.tail(8).to_string())

BEV series: 45 quarters
Range: 2014-09-30 → 2025-09-30
                               Date  bev_cumulative  quarter
date                                                        
2023-12-31   2023 Q4 (end December)          930649  2023 Q4
2024-03-31      2024 Q1 (end March)         1009536  2024 Q1
2024-06-30       2024 Q2 (end June)         1089220  2024 Q2
2024-09-30  2024 Q3 (end September)         1191066  2024 Q3
2024-12-31   2024 Q4 (end December)         1287198  2024 Q4
2025-03-31      2025 Q1 (end March)         1402636  2025 Q1
2025-06-30       2025 Q2 (end June)         1504927  2025 Q2
2025-09-30  2025 Q3 (end September)         1629450  2025 Q3


In [46]:
from pathlib import Path
import numpy as np

DATA_DIR = Path('../data')
MANDATE_DATE = pd.Timestamp('2022-06-01')

# Interpolate to monthly, then daily
monthly = bev['bev_cumulative'].resample('ME').interpolate('linear')

# Compliant stock = new BEVs registered after mandate × compliance rate
# Assumption: each post-mandate BEV gets a smart charger
COMPLIANCE_RATE = 0.50
CHARGER_PER_BEV = 0.85  # ~85% of new BEV buyers install a home charger

baseline = monthly.loc[monthly.index <= MANDATE_DATE].iloc[-1]

compliant = np.where(
    monthly.index > MANDATE_DATE,
    (monthly - baseline).clip(lower=0) * CHARGER_PER_BEV * COMPLIANCE_RATE,
    0
)
compliant_monthly = pd.Series(compliant, index=monthly.index, name='compliant_stock')

# Save as parquet
cp_df = compliant_monthly.to_frame()
cp_df['bev_cumulative'] = monthly
cp_df.to_parquet(DATA_DIR / 'chargepoints.parquet')

print(f'Saved → {DATA_DIR}/chargepoints.parquet')
print(f'Quarters: {len(bev)}  |  Monthly interpolated: {len(cp_df)}')
print(f'\nCompliant stock at latest: {cp_df.compliant_stock.iloc[-1]:,.0f}')
print(f'BEVs at latest:            {cp_df.bev_cumulative.iloc[-1]:,.0f}')
print(f'\nShiftable load estimate:')
gw = cp_df.compliant_stock.iloc[-1] * 7 * 0.30 / 1e6
print(f'  {cp_df.compliant_stock.iloc[-1]:,.0f} chargepoints × 7kW × 30% simultaneity = {gw:.2f} GW')

Saved → ../data/chargepoints.parquet
Quarters: 45  |  Monthly interpolated: 133

Compliant stock at latest: 490,575
BEVs at latest:            1,629,450

Shiftable load estimate:
  490,575 chargepoints × 7kW × 30% simultaneity = 1.03 GW


In [47]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../data')

files = {
    'prices_raw.parquet':         'GB day-ahead prices (Elexon MID)',
    'demand_raw.parquet':         'GB half-hourly demand ND/TSD (NESO)',
    'wind_total_raw.parquet':     'Total wind actuals + DA forecast error',
    'interconnector_raw.parquet': 'Net interconnector flows (NESO)',
    'chargepoints.parquet':       'Compliant chargepoint stock (DVLA-derived)',
}

print('=== DATA INVENTORY ===\n')
all_ready = True
for fname, desc in files.items():
    fpath = DATA_DIR / fname
    if fpath.exists():
        df = pd.read_parquet(fpath)
        print(f'  ✓  {fname:<35} {len(df):>8,} rows')
        print(f'       {desc}')
    else:
        print(f'  ✗  {fname:<35} MISSING')
        all_ready = False

print()
if all_ready:
    print('All files present. Ready for Notebook 02.')
else:
    print('Some files missing.')

=== DATA INVENTORY ===

  ✓  prices_raw.parquet                   127,155 rows
       GB day-ahead prices (Elexon MID)
  ✓  demand_raw.parquet                    73,536 rows
       GB half-hourly demand ND/TSD (NESO)
  ✓  wind_total_raw.parquet               127,160 rows
       Total wind actuals + DA forecast error
  ✓  interconnector_raw.parquet           126,192 rows
       Net interconnector flows (NESO)
  ✓  chargepoints.parquet                     133 rows
       Compliant chargepoint stock (DVLA-derived)

All files present. Ready for Notebook 02.


## 4. Battery Storage Capacity — ESO Generation Mix

Tracks installed GB battery storage (MW) over time.  
Used as a confound control and for the pre-2021 falsification test.

In [ ]:
STORAGE_FILE = DATA_DIR / 'storage_raw.parquet'

# ESO Historic Generation Mix includes battery storage
# Resource: https://data.nationalgrideso.com/carbon-intensity1/historic-generation-mix
ESO_GENMIX_RESOURCE = 'b82b9e4a-e966-4d62-b119-76e45c6e9e2f'

if STORAGE_FILE.exists():
    print(f'Cache hit: {STORAGE_FILE}')
    storage = pd.read_parquet(STORAGE_FILE)
else:
    print('Fetching generation mix (for battery storage)...')
    genmix = fetch_eso_resource(ESO_GENMIX_RESOURCE)
    genmix.columns = [c.lower().replace(' ', '_') for c in genmix.columns]
    
    # Find battery column
    bat_col = next((c for c in genmix.columns if 'batter' in c or 'storage' in c), None)
    print('Battery column:', bat_col)
    
    dt_col = next(c for c in genmix.columns if 'date' in c or 'time' in c)
    genmix['datetime'] = pd.to_datetime(genmix[dt_col], utc=True)
    storage = (
        genmix[['datetime', bat_col]]
        .rename(columns={bat_col: 'battery_mw'})
        .set_index('datetime')
        .sort_index()
    )
    storage.to_parquet(STORAGE_FILE)
    print(f'Saved → {STORAGE_FILE}')

print(f'Storage data: {len(storage):,} rows')
print(f'Battery MW range: {storage.battery_mw.min():.0f} → {storage.battery_mw.max():.0f}')

## 5. EV Registrations — DVLA

Quarterly cumulative BEV registrations.  
Download the CSV from DVLA and place at `../data/ev_registrations_raw.csv`.  

**Source:** https://www.gov.uk/government/statistical-data-sets/vehicle-licensing-statistics  
Table VEH0131 — Licensed vehicles by body type and propulsion / fuel type

Expected columns: `quarter` (e.g. `2023 Q4`), `bev_cumulative`

In [ ]:
EV_FILE     = DATA_DIR / 'ev_registrations_raw.csv'
EV_OUT_FILE = DATA_DIR / 'ev_registrations.parquet'

def parse_dvla_ev(path: Path) -> pd.DataFrame:
    """
    Parse DVLA VEH0131 table.
    Returns monthly interpolated cumulative BEV stock with datetime index.
    """
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    
    # Identify the quarter column and BEV column
    q_col  = next(c for c in df.columns if 'quarter' in c or 'period' in c)
    bev_col = next(c for c in df.columns if 'bev' in c or 'electric' in c or 'battery' in c)
    
    df = df[[q_col, bev_col]].dropna()
    df.columns = ['quarter', 'bev_cumulative']
    
    # Parse quarter strings → period end date
    df['date'] = pd.PeriodIndex(df['quarter'], freq='Q').to_timestamp('Q')
    df = df.set_index('date').sort_index()
    df['bev_cumulative'] = pd.to_numeric(
        df['bev_cumulative'].astype(str).str.replace(',', ''), errors='coerce'
    )
    
    # Interpolate to monthly
    monthly = df['bev_cumulative'].resample('ME').interpolate('linear')
    return monthly.to_frame()


if EV_OUT_FILE.exists():
    print(f'Cache hit: {EV_OUT_FILE}')
    ev = pd.read_parquet(EV_OUT_FILE)
elif EV_FILE.exists():
    print('Parsing DVLA EV registrations...')
    ev = parse_dvla_ev(EV_FILE)
    ev.to_parquet(EV_OUT_FILE)
    print(f'Saved → {EV_OUT_FILE}')
else:
    print('⚠️  DVLA file not found.')
    print(f'   Place VEH0131 CSV at: {EV_FILE.resolve()}')
    print('   Download: https://www.gov.uk/government/statistical-data-sets/vehicle-licensing-statistics')
    ev = None

if ev is not None:
    print(f'EV data: {len(ev)} months | latest: {ev.bev_cumulative.iloc[-1]:,.0f} BEVs')

## 6. Chargepoint Installations — OZEV

Quarterly public/home chargepoint installation data.  
Download from OZEV and place at `../data/chargepoints_raw.csv`.

**Source:** https://www.gov.uk/government/statistics/electric-vehicle-charging-device-statistics  
Table CHR1101 or equivalent — cumulative installations by device type

Expected columns: `quarter`, `home_chargepoints_cumulative`

In [ ]:
CP_FILE     = DATA_DIR / 'chargepoints_raw.csv'
CP_OUT_FILE = DATA_DIR / 'chargepoints.parquet'

def build_compliant_stock(
    cp_df: pd.DataFrame,
    mandate_date: str = '2022-06-01',
    compliance_rate: float = 0.50,
) -> pd.DataFrame:
    """
    Build cumulative compliant chargepoint stock.
    
    Mandate: from June 2022, new home chargepoints must be smart.
    Treatment = cumulative installations post-mandate × compliance rate.
    Pre-mandate stock is pre-treatment (control period).
    """
    mandate_ts = pd.Timestamp(mandate_date)
    cp_df = cp_df.copy()
    cp_df['post_mandate'] = cp_df.index >= mandate_ts
    
    # Stock at mandate date (baseline)
    baseline = cp_df.loc[cp_df.index < mandate_ts, 'home_chargepoints_cumulative'].iloc[-1] if len(cp_df.loc[cp_df.index < mandate_ts]) > 0 else 0
    
    cp_df['incremental_post_mandate'] = np.where(
        cp_df['post_mandate'],
        (cp_df['home_chargepoints_cumulative'] - baseline).clip(lower=0),
        0,
    )
    cp_df['compliant_stock'] = cp_df['incremental_post_mandate'] * compliance_rate
    return cp_df


def parse_ozev_chargepoints(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    q_col  = next(c for c in df.columns if 'quarter' in c or 'period' in c)
    cp_col = next(c for c in df.columns if 'home' in c or 'domestic' in c or 'install' in c)
    df = df[[q_col, cp_col]].dropna()
    df.columns = ['quarter', 'home_chargepoints_cumulative']
    df['date'] = pd.PeriodIndex(df['quarter'], freq='Q').to_timestamp('Q')
    df = df.set_index('date').sort_index()
    df['home_chargepoints_cumulative'] = pd.to_numeric(
        df['home_chargepoints_cumulative'].astype(str).str.replace(',', ''), errors='coerce'
    )
    monthly = df['home_chargepoints_cumulative'].resample('ME').interpolate('linear').to_frame()
    monthly = build_compliant_stock(monthly)
    return monthly


if CP_OUT_FILE.exists():
    print(f'Cache hit: {CP_OUT_FILE}')
    chargepoints = pd.read_parquet(CP_OUT_FILE)
elif CP_FILE.exists():
    print('Parsing OZEV chargepoint data...')
    chargepoints = parse_ozev_chargepoints(CP_FILE)
    chargepoints.to_parquet(CP_OUT_FILE)
    print(f'Saved → {CP_OUT_FILE}')
else:
    print('⚠️  OZEV file not found.')
    print(f'   Place OZEV chargepoint CSV at: {CP_FILE.resolve()}')
    print('   Download: https://www.gov.uk/government/statistics/electric-vehicle-charging-device-statistics')
    chargepoints = None

if chargepoints is not None:
    print(f'Chargepoint data: {len(chargepoints)} months')
    print(f'Compliant stock at latest: {chargepoints.compliant_stock.iloc[-1]:,.0f}')

## 7. Pipeline Summary

Quick inventory of what's cached and ready.

In [ ]:
import os

print('=== DATA PIPELINE INVENTORY ===')
print()

expected_files = {
    'prices_raw.parquet':       'GB day-ahead prices (Elexon)',
    'wind_raw.parquet':         'Wind actuals + DA forecasts (ESO)',
    'demand_raw.parquet':       'GB half-hourly demand (ESO)',
    'interconnector_raw.parquet': 'Net interconnector flows (Elexon)',
    'storage_raw.parquet':      'Battery storage capacity (ESO)',
    'ev_registrations.parquet': 'EV registrations (DVLA)',
    'chargepoints.parquet':     'Chargepoint installations (OZEV)',
}

all_ready = True
for fname, desc in expected_files.items():
    fpath = DATA_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1e6
        print(f'  ✓  {fname:<40} {size_mb:.1f} MB   {desc}')
    else:
        print(f'  ✗  {fname:<40} MISSING        {desc}')
        all_ready = False

print()
if all_ready:
    print('All files present. Proceed to Notebook 02.')
else:
    print('Some files missing. See cells above for download instructions.')